# 4장 복습과제 — GBM과 XGBoost

**범위:** 교재 4.5 GBM, 4.6 XGBoost  
**문항:** 코딩 문제 5개  
**방법:** 데이터 준비 셀을 먼저 실행하고, 각 문제 아래의 빈 코드 셀에 코드를 작성하세요. 문제는 위에서 아래 순서대로 이어집니다.  
**참고:** `xgboost` 패키지가 없는 환경이라면 설치한 다음 실행하세요. 정확도와 조기 중단 시점은 실행 환경에 따라 달라질 수 있습니다.


## 데이터 준비 — 그대로 실행

위스콘신 유방암 데이터의 레이블은 `0=악성`, `1=양성`입니다. 전체 데이터에서 학습용 80%와 테스트용 20%를 분리하고, 학습용을 다시 학습·검증 데이터로 나눕니다.


In [1]:
import numpy as np
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=156, stratify=y
)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=156, stratify=y_train
)

print('전체 데이터:', X.shape)
print('학습:', X_tr.shape, '검증:', X_val.shape, '테스트:', X_test.shape)
print('레이블:', dict(enumerate(cancer.target_names)))


전체 데이터: (569, 30)
학습: (409, 30) 검증: (46, 30) 테스트: (114, 30)
레이블: {0: np.str_('malignant'), 1: np.str_('benign')}


## 문제 1. GBM 모델 학습과 정확도 평가

`GradientBoostingClassifier`로 유방암 데이터를 분류하세요.

1. `n_estimators=100`, `learning_rate=0.1`, `max_depth=3`, `random_state=42`로 모델 생성
2. **`X_train`, `y_train`**으로 학습하고 `X_test` 예측
3. 테스트 정확도를 소수점 넷째 자리까지 출력


In [2]:
gb_clf=GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_clf.fit(X_train, y_train)
gb_pred= gb_clf.predict(X_test)
gb_accuracy=accuracy_score(y_test, gb_pred)
print('GBM 정확도: {0:.4f}'.format(gb_accuracy))

GBM 정확도: 0.9825


## 문제 2. 파이썬 래퍼용 `DMatrix` 만들기

학습·검증·테스트 데이터를 `xgb.DMatrix`로 각각 변환하세요.

1. **`X_tr`, `y_tr`**로 `dtr` 생성
2. **`X_val`, `y_val`**로 `dval` 생성
3. **`X_test`, `y_test`**로 `dtest` 생성
4. `num_row()`를 이용해 각각의 데이터 건수 출력


In [3]:
dtr=xgb.DMatrix(data=X_tr, label=y_tr)
dval= xgb.DMatrix(data=X_val, label=y_val)
dtest= xgb.DMatrix(data=X_test, label=y_test)

## 문제 3. 파이썬 래퍼 XGBoost 학습과 조기 중단

문제 2의 `dtr`과 `dval`로 모델을 학습하세요.

1. `params` 딕셔너리에 `max_depth=3`, `eta=0.05`, `objective='binary:logistic'`, `eval_metric='logloss'`, `seed=42` 지정
2. 평가 목록 `[(dtr, 'train'), (dval, 'eval')]`을 만들어 학습·검증 데이터 지정
3. `xgb.train()`에 `num_boost_round=200`, `early_stopping_rounds=20` 적용
4. 학습 완료 후 모델의 `best_iteration`과 `best_score` 출력

학습 로그가 너무 길면 `verbose_eval=False`를 추가해도 됩니다.


In [4]:
params= {'max_depth':3, 'eta':0.05, 'objective':'binary:logistic', 'eval_metirc':'logloss', 'seed':42}
eval_list = [(dtr, 'train'), (dval, 'eval')]
xgb_model = xgb.train(params=params, dtrain=dtr, evals=eval_list, num_boost_round=200, early_stopping_rounds=20)
pred_probs=xgb_model.predict(dtest)
print('가장 성능이 좋은 iteration:', xgb_model.best_iteration)
print('그때의 평가 점수:', xgb_model.best_score)


[0]	train-logloss:0.61854	eval-logloss:0.61768
[1]	train-logloss:0.58088	eval-logloss:0.58288
[2]	train-logloss:0.54617	eval-logloss:0.55031
[3]	train-logloss:0.51463	eval-logloss:0.52069
[4]	train-logloss:0.48563	eval-logloss:0.49278
[5]	train-logloss:0.45956	eval-logloss:0.46862
[6]	train-logloss:0.43502	eval-logloss:0.44596
[7]	train-logloss:0.41237	eval-logloss:0.42475
[8]	train-logloss:0.39144	eval-logloss:0.40633
[9]	train-logloss:0.37213	eval-logloss:0.38860
[10]	train-logloss:0.35405	eval-logloss:0.37175


c:\Users\0510n\anaconda3\Lib\site-packages\xgboost\callback.py:385: UserWarning: [02:11:29] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "eval_metirc" } are not used.

  self.starting_round = model.num_boosted_rounds()


[11]	train-logloss:0.33707	eval-logloss:0.35522
[12]	train-logloss:0.32096	eval-logloss:0.34086
[13]	train-logloss:0.30664	eval-logloss:0.32720
[14]	train-logloss:0.29255	eval-logloss:0.31473
[15]	train-logloss:0.27987	eval-logloss:0.30380
[16]	train-logloss:0.26810	eval-logloss:0.29257
[17]	train-logloss:0.25641	eval-logloss:0.28142
[18]	train-logloss:0.24552	eval-logloss:0.27203
[19]	train-logloss:0.23579	eval-logloss:0.26274
[20]	train-logloss:0.22613	eval-logloss:0.25534
[21]	train-logloss:0.21716	eval-logloss:0.24800
[22]	train-logloss:0.20903	eval-logloss:0.24024
[23]	train-logloss:0.20088	eval-logloss:0.23392
[24]	train-logloss:0.19335	eval-logloss:0.22780
[25]	train-logloss:0.18618	eval-logloss:0.22211
[26]	train-logloss:0.17929	eval-logloss:0.21694
[27]	train-logloss:0.17289	eval-logloss:0.21137
[28]	train-logloss:0.16693	eval-logloss:0.20718
[29]	train-logloss:0.16090	eval-logloss:0.20268
[30]	train-logloss:0.15553	eval-logloss:0.19787
[31]	train-logloss:0.15048	eval-logloss:

## 문제 4. 예측 확률을 클래스로 바꾸고 평가하기

문제 3의 파이썬 래퍼 모델로 테스트 데이터를 예측하세요.

1. `predict(dtest)`로 **클래스 1의 예측 확률** 계산
2. 확률이 **0.5보다 크면 1**, 그렇지 않으면 **0**으로 변환
3. 예측 확률 앞 5개와 변환한 클래스 앞 5개 출력
4. `accuracy_score()`로 테스트 정확도를 소수점 넷째 자리까지 출력

확률을 클래스로 변환할 때는 리스트 컴프리헨션이나 `np.where()` 중 편한 방법을 사용하세요.


In [ ]:
pred_probs=xgb_model.predict(dtest) #클래스 1의 예측 확률 반환 
wp_class=[1 if x>0.5 else 0 for x in pred_probs] #클래스 분류 
acc= accuracy_score(y_test, wp_class) 

print('예측 확률 앞 5개:',pred_probs[:5])
print('변환한 클래스 앞 5개:', wp_class[:5])
print('파이썬 래퍼 테스트 정확도: {:.4f}'.format(acc))

예측 확률 앞 5개: [0.00198853 0.9975642  0.9854474  0.9988263  0.9826736 ]
변환한 클래스 앞 5개: [0, 1, 1, 1, 1]
파이썬 래퍼 테스트 정확도: 0.9912


In [ ]:
print(pred_probs)
type(pred_probs)

[0.00198853 0.9975642  0.9854474  0.9988263  0.9826736  0.98284507
 0.99725974 0.99862325 0.9968636  0.99765766 0.9925484  0.00181542
 0.00776288 0.00146582 0.9943295  0.93141514 0.9981883  0.00123925
 0.9986933  0.99910444 0.9660916  0.00213855 0.9950551  0.00874918
 0.00143141 0.99893326 0.5805488  0.00540159 0.99916875 0.98238623
 0.02099632 0.00346775 0.00230937 0.8848318  0.00181877 0.9981061
 0.99581474 0.9947212  0.00328302 0.00184393 0.84270376 0.9933356
 0.04821859 0.9976804  0.00140788 0.00206444 0.06730244 0.99813557
 0.9981475  0.998044   0.07642993 0.00442906 0.99897397 0.00117098
 0.9976349  0.9906386  0.99886703 0.9985703  0.9967982  0.7057686
 0.0012858  0.99852604 0.9983369  0.98802966 0.99495065 0.9972492
 0.99868256 0.99862444 0.99405056 0.9984628  0.9118531  0.00114515
 0.9867019  0.0031569  0.9971405  0.799127   0.99527454 0.99801695
 0.99112904 0.00202169 0.9976659  0.00334538 0.44461414 0.9966102
 0.99078256 0.99819    0.0021512  0.9984805  0.0017827  0.06561082


numpy.ndarray

## 문제 5. 사이킷런 래퍼 `XGBClassifier` 적용

이번에는 사이킷런 래퍼로 유방암 데이터를 분류하세요.

1. `n_estimators=200`, `learning_rate=0.05`, `max_depth=3`, `objective='binary:logistic'`, `eval_metric='logloss'`, `random_state=42`로 모델 생성
2. **`X_tr`, `y_tr`**로 학습하고 `X_test`에 대해 `predict()` 실행
3. `predict_proba()`에서 **클래스 1의 예측 확률**만 추출
4. 클래스 1의 예측 확률 앞 5개와 테스트 정확도를 소수점 넷째 자리까지 출력

이번 문제에서는 조기 중단을 적용하지 않습니다.


In [14]:
xgb_wrapper= XGBClassifier(n_estimators=200, learing_rate=0.05, max_depth=3, objective='binary:logistic', eval_metric='logloss', random_state=42)
xgb_wrapper.fit(X_tr, y_tr)
pred= xgb_wrapper.predict(X_test)
pred_proba= xgb_wrapper.predict_proba(X_test)[:, 1]

print('클래스 1의 예측 확률 앞 5개:', np.round(pred_proba[:5], 3)) 
accuracy = accuracy_score(y_test, pred)
print('사이킷런 래퍼 테스트 정확도: {:.4f}'.format(accuracy))


c:\Users\0510n\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [02:23:23] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "learing_rate" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


클래스 1의 예측 확률 앞 5개: [0.001 1.    0.999 1.    0.987]
사이킷런 래퍼 테스트 정확도: 0.9912


# **LightGBM**
* Heart Failure Clinical Records(심부전 환자 사망 여부 예측) 데이터셋을 활용합니다.
* 환자의 나이, 빈혈 여부, 당뇨 여부, 박출률, 혈청 크레아티닌, 혈청 나트륨, 고혈압 여부, 흡연 여부 등 여러 임상 정보를 바탕으로 생존(0)인지 사망(1)인지 분류합니다.
* 아래 링크에서 데이터셋을 다운 받아주세요.

https://www.kaggle.com/datasets/andrewmvd/heart-failure-clinical-data

* \## 답안 ## 표시된 부분을 채워서 코드를 완성해주세요.

# **데이터 처리**

# 문제 6. 데이터 로딩 및 확인

아래 코드에서 경로를 설정해 데이터를 불러오고,  클래스(`DEATH_EVENT`) 값 분포를 확인해주세요.

In [30]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

heart_df = pd.read_csv('./heart_failure_clinical_records_dataset.csv') # 경로 확인

print(heart_df['DEATH_EVENT'].value_counts()) # 클래스 값 분포 확인(피처명: 'DEATH_EVENT')

DEATH_EVENT
0    203
1     96
Name: count, dtype: int64


In [31]:
# 데이터셋 살펴보기
heart_df.head(3)

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1


# 문제 7. 피처/레이블 분리 및 데이터 분리

* `DEATH_EVENT` 칼럼을 레이블로, 나머지를 피처로 분리해주세요.
* 전체 데이터 중 80%는 학습용, 20%는 테스트용으로 추출해주세요. (`random_state=156`)
* 위에서 만든 학습용 데이터를 다시 90%는 학습용, 10%는 검증용으로 분리해주세요. (`random_state=156`)

In [33]:
from sklearn.model_selection import train_test_split

X_features = heart_df.iloc[:,:-1]
y_label = heart_df['DEATH_EVENT']

# 전체 데이터 중 80%는 학습용 데이터, 20%는 테스트용 데이터 추출
X_train, X_test, y_train, y_test = train_test_split(X_features, y_label, test_size=0.2)

# 위에서 만든 학습용 데이터를 다시 쪼개서 90%는 학습, 10%는 검증용 데이터로 분리
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.1)

# **LightGBM 적용**

# 문제 8. LightGBM 모델 학습하기

lightgbm 패키지의 LGBMClassifier를 이용하여 학습을 수행해주세요.
* `n_estimators`: 350
* `learning_rate`: 0.05
* 조기중단(`early stopping`) 라운드: 50
* 평가지표: logloss

**참고**: 교재 코드와 달리, 최신 버전의 LightGBM은 조기 중단을 수행할 때 `fit()`의 `early_stopping_rounds` 파라미터를 직접 사용할 수 없습니다. 대신 `callbacks` 파라미터에 `early_stopping()`함수를 리스트로 전달해야 합니다.

In [34]:
# 아직 설치되어 있지 않다면 아래 셀을 실행해 설치해 주세요.
!pip install lightgbm

^C


   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 7.8 MB/s  0:00:00



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\0510n\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [37]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

lgbm_wrapper = LGBMClassifier(n_estimaotrs=350, learning_rate=0.05)

evals = [(X_tr, y_tr), (X_val, y_val)]
lgbm_wrapper.fit(X_tr, y_tr, eval_metric ='logloss', eval_set=evals,
                 callbacks=[early_stopping(stopping_rounds=50), log_evaluation(period=10)])

preds = lgbm_wrapper. predict(X_test)
pred_proba = lgbm_wrapper.predict_proba(X_test)[:, 1]

[LightGBM] [Warning] Unknown parameter: n_estimaotrs
Training until validation scores don't improve for 50 rounds
[10]	training's binary_logloss: 0.476432	valid_1's binary_logloss: 0.427692
[20]	training's binary_logloss: 0.385934	valid_1's binary_logloss: 0.350575
[30]	training's binary_logloss: 0.325993	valid_1's binary_logloss: 0.317532
[40]	training's binary_logloss: 0.279855	valid_1's binary_logloss: 0.299128
[50]	training's binary_logloss: 0.244994	valid_1's binary_logloss: 0.290316
[60]	training's binary_logloss: 0.21947	valid_1's binary_logloss: 0.283449
[70]	training's binary_logloss: 0.194081	valid_1's binary_logloss: 0.287621
[80]	training's binary_logloss: 0.172177	valid_1's binary_logloss: 0.281423
[90]	training's binary_logloss: 0.15436	valid_1's binary_logloss: 0.28071
[100]	training's binary_logloss: 0.139396	valid_1's binary_logloss: 0.28557
Did not meet early stopping. Best iteration is:
[100]	training's binary_logloss: 0.139396	valid_1's binary_logloss: 0.28557


# 문제 9. 성능 평가하기
교재에서 사용된 `get_clf_eval` 함수입니다. 함수를 실행하고, 그 아래 코드에 알맞은 파라미터를 넣어 성능 평가를 수행해주세요.

In [38]:
from sklearn.metrics import (accuracy_score, precision_score,
  recall_score, f1_score, roc_auc_score, confusion_matrix)

def get_clf_eval(y_test, pred=None, pred_proba=None):
  confusion = confusion_matrix(y_test, pred)
  accuracy = accuracy_score(y_test, pred)
  precision = precision_score(y_test, pred)
  recall = recall_score(y_test, pred)
  f1 = f1_score(y_test, pred)
  roc_auc = roc_auc_score(y_test, pred_proba)
  print('오차 행렬')
  print(confusion)
  print('정확도:{:.4f}, 정밀도: {:.4f}, 재현율:{:.4f},\
  F1:{:.4f}, AUC: {:.4f}'.format(accuracy, precision, recall, f1, roc_auc))

get_clf_eval(y_test, preds, pred_proba)

오차 행렬
[[41  3]
 [ 3 13]]
정확도:0.9000, 정밀도: 0.8125, 재현율:0.8125,  F1:0.8125, AUC: 0.9403


# **베이지안 최적화 기반의 HyperOpt를 이용한 하이퍼 파라미터 튜닝**

# **HyperOpt 사용하기**

# 문제 10. HyperOpt 기본 사용법 익히기
아래 조건에 맞게 HyperOpt의 hp 모듈로 검색 공간을 설정하고, search_space에서 x, y 입력 변숫값을 추출하여 `retval = x**2 - 20*y`로 계산된 값을 반환하는 목적 함수를 완성해주세요.
* 입력 변수 x: -10부터 10까지 1 간격
* 입력 변수 y: -15부터 15까지 1 간격

In [40]:
# 아직 설치되어 있지 않다면 아래 셀을 실행해 설치해 주세요.
!pip install hyperopt

  Using cached hyperopt-0.3.0-py3-none-any.whl.metadata (9.5 kB)
Using cached hyperopt-0.3.0-py3-none-any.whl (973 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 10.1 MB/s  0:00:00

   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ------------- -------------------------- 1/3 [networkx]
   ------------- -------------------------- 1/3 [networkx]
   ------------- -------------------------- 1/3 [networkx]
   ------------- -------------------------- 1/3 [networkx]
   ------------- -------------------------- 1/3 [networkx]
   ------------- -------------------------- 1/3 [networkx]
   ------------- -------------------------- 1/3 [networkx]
   ------------- -------------------------- 1/3 [


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\0510n\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [46]:
from hyperopt import hp, STATUS_OK

search_space = {'x':hp.quniform('x', -10, 10, 1), 'y':hp.quniform('y', -15, 15, 1)}

def objective_func(search_space):
  x = search_space['x']
  y = search_space['y']
  retval = x**2-20*y

  return retval



# 문제 11. fmin()으로 최적 입력값 찾기
fmin() 함수를 이용해 위에서 설정한 목적 함수의 반환값이 최솟값이 되는 최적 입력 변숫값을 찾아주세요.
* `fn`: 위에서 설정한 목적 함수
* `sapce`: 위에서 설정한 검색 공산
* `algo`: tpe.suggest
* `max_evals`: 20
* `trials`: Trials() 객체
* `rstate=np.random.default_rng(seed=0)`

In [47]:
from hyperopt import fmin, tpe, Trials

trial_val = Trials()

best_01 = fmin(fn=objective_func,
               space=search_space,
               algo=tpe.suggest,
               max_evals=20,
               trials=trial_val,
               rstate=np.random.default_rng(seed=0))
print('best:', best_01)

100%|██████████| 20/20 [00:00<00:00, 721.12trial/s, best loss: -296.0]
best: {'x': np.float64(2.0), 'y': np.float64(15.0)}


# **HyperOpt를 이용한 XGBoost 하이퍼 파라미터 최적화**

여기서부터는 문제 6~9에서 만든 Heart Failure Clinical Records(심부전 환자 사망 여부 예측) 데이터셋(X_train, y_train)을 다시 사용합니다.

# 문제 12. 검색 공간 및 목적 함수 설정하기
아래 조건에 맞게 XGBoost 하이퍼 파라미터 검색 공간을 설정하고, 목적 함수를 완성해주세요.

**검색공간**
* max_depth: 4에서 12까지 1 간격
* min_child_weight: 1에서 5까지 1 간격
* learning_rate: 0.01에서 0.15 사이 정규 분포된 값
* colsample_bytree: 0.5에서 0.9 사이 정규 분포된 값


**목적 함수** (교차 검증 cv=3 기반의 평균 정확도(accuracy)를 반환)
1. 검색 공간에서 목적 함수로 입력되는 모든 값은 실수형이므로, 정수형이어야 하는 하이퍼 파라미터는 형변환을 해주세요.
2. HyperOpt의 목적 함수는 최솟값을 반환하도록 최적화해야 하기 때문에, 정확도처럼 값이 클수록 좋은 성능 지표일 경우 -1을 곱한 뒤 반환해야 합니다.

In [48]:
# 아직 설치되어 있지 않다면 아래 셀을 실행해 설치해 주세요.
!pip install xgboost

  Using cached xgboost-3.4.1-py3-none-win_amd64.whl.metadata (2.0 kB)
Using cached xgboost-3.4.1-py3-none-win_amd64.whl (48.9 MB)



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\0510n\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [56]:
from typing_extensions import DefaultDict
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
from hyperopt import STATUS_OK

xgb_search_space = {'max_depth': hp.quniform('max_depth', 4, 12, 1),
                    'min_child_weight': hp.quniform('min_child_weight', 1, 5, 1),
                    'learning_rate': hp.uniform('learning_rate', 0.01, 0.15),
                    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 0.9)}

def objective_func(search_space):
  xgb_clf = XGBClassifier(n_estimaors=100,
                          max_depth=int(search_space['max_depth']),
                          min_child_weight=int(search_space['min_child_weight']),
                          learning_rate=search_space['learning_rate'],
                          colsample_bytree=search_space['colsample_bytree'],
                          eval_metric='logloss')
  accuracy = cross_val_score(xgb_clf, X_train, y_train, scoring='accuracy', cv=3)

  return {'loss': -1 * np.mean(accuracy), 'status': STATUS_OK }

# 문제 13. fmin()으로 최적 하이퍼 파라미터 도출 및 재학습

1. `max_eval=50`, `rstate=np.random.default_rng(seed=9)`으로 `fmin()`을 호출해 최적 하이퍼 파라미터를 찾아주세요.
2. 도출된 최적 하이퍼 파라미터로 XGBClassifier를 재학습해주세요.
- `n_estimators`:350
- 조기 중단 라운드: 50
- 평가지표: logloss
- 검증 데이터: (X_tr, y_tr), (X_val, y_val)
- 정수형/실수형 하이퍼 파라미터의 형변환에 유의하세요.
3. 문제 3에서 만든 `get_clf_eval()` 함수로 테스트 데이터 세트에 대한 성능을 평가해주세요.


**참고**: 교재 코드와 달리, 최신 버전의 XGBoost에서는 `fit()`의 `eval_metric`, `early_stopping_rounds`, `callbacks` 파라미터를 직접 사용할 수 없습니다. 대신 해당 파라미터들을 `XGBClassifier()` 생성 시 전달해야 하며, `fit()`에는 `eval_set`과 `verbose` 등을 전달합니다.

In [57]:
from hyperopt import fmin, tpe, Trials

trial_val = Trials()
best = fmin(fn=objective_func,
            space=xgb_search_space,
            algo=tpe.suggest,
            max_evals=50,
            trials=trial_val,
            rstate=np.random.default_rng(seed=9))
print('best:', best)

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

100%|██████████| 50/50 [00:31<00:00,  1.57trial/s, best loss: -0.8452004219409283]
best: {'colsample_bytree': np.float64(0.8817066584767306), 'learning_rate': np.float64(0.026468945671419834), 'max_depth': np.float64(11.0), 'min_child_weight': np.float64(5.0)}


In [59]:
#도출된 최적의 하이퍼 파라미터를 직접 classifier 에 다시 넣고, 형변환 후 확인 
xgb_wrapper = XGBClassifier(n_estimators=350,
                            learning_rate=round(best['learning_rate'], 5),  # 소수점 아래 5자리까지 반올림
                            max_depth=int(best['max_depth']),
                            min_child_weight=int(best['min_child_weight']),
                            colsample_bytree=best['colsample_bytree'],  # 소수점 아래 5자리까지 반올림
                            eval_metric='logloss',
                            early_stopping_rounds=50)

evals = [(X_tr, y_tr), (X_val, y_val)]
xgb_wrapper.fit(X_tr, y_tr, eval_set=evals, verbose=True)

preds = xgb_wrapper.predict(X_test)
pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]

[0]	validation_0-logloss:0.63173	validation_1-logloss:0.57117
[1]	validation_0-logloss:0.62072	validation_1-logloss:0.56020
[2]	validation_0-logloss:0.61025	validation_1-logloss:0.54916
[3]	validation_0-logloss:0.60039	validation_1-logloss:0.53925
[4]	validation_0-logloss:0.59123	validation_1-logloss:0.53048
[5]	validation_0-logloss:0.58213	validation_1-logloss:0.52149
[6]	validation_0-logloss:0.57342	validation_1-logloss:0.51328
[7]	validation_0-logloss:0.56528	validation_1-logloss:0.50487
[8]	validation_0-logloss:0.55660	validation_1-logloss:0.49646
[9]	validation_0-logloss:0.54914	validation_1-logloss:0.48881
[10]	validation_0-logloss:0.54190	validation_1-logloss:0.48174
[11]	validation_0-logloss:0.53492	validation_1-logloss:0.47496
[12]	validation_0-logloss:0.52815	validation_1-logloss:0.46803
[13]	validation_0-logloss:0.52108	validation_1-logloss:0.46238
[14]	validation_0-logloss:0.51498	validation_1-logloss:0.45674
[15]	validation_0-logloss:0.50857	validation_1-logloss:0.45163
[1

In [63]:
get_clf_eval(y_test, preds, pred_proba)

오차 행렬
[[40  4]
 [ 3 13]]
정확도:0.8833, 정밀도: 0.7647, 재현율:0.8125,  F1:0.7879, AUC: 0.9290


# 문제 14. 질문에 대한 답안을 작성해주세요

**1. LightGBM은 일반 GBM 계열의 트리 분할 방법과 다르게 어떤 방식을 사용하나요? 그 방식의 장단점을 간략히 서술하세요.**

트리의 균형을 맞추지 않고, 최대 손실값을 가지는 리프 노드를 지속적으로 분할하면서 트리의 깊이가 깊어지고 비대칭적인 규칙 트리가 생성된다. 
장점: 일반적인 균형트리방식보다 예측 오류 손실 최소화
단점: 적은 데이터 세트에 적용할 경우 과적합 발생이 쉬움 

**2. 베이지안 최적화가 Grid Search 방식에 비해 갖는 장점을 대체 모델과 획득 함수의 역할을 포함하여 서술하세요.**

베이지안 최적화는 Grid Search보다 적은 횟수의 시도로 최적의 하이퍼파라미터를 찾을 수 있다는 장점이 있다. 
Grid Search는 모든 하이퍼파라미터 조합을 직접 학습하지만, 베이지안 최적화는 대체 모델과 획득 함수를 이용한다. 대체 모델이 기존 실험 결과를 바탕으로 성능을 예측하고, 획득 함수가 다음에 평가할 하이퍼파라미터 조합을 선택한다. 이를 반복하면서 효율적으로 최적값을 찾아간다.

# 분류 실습과 스태킹 앙상블

**범위:** 교재 4.5 분류 실습, 4.6 스태킹 앙상블  
**문항:** 코딩 문제 5개  
**준비:** 캐글 신용카드 사기 검출 실습 코드를 띄워놓고 문제를 풀어주세요.  



## 문제 15. GradientBoostingClassifier을 이용한 학습

사이킷런의 GradientBoostingClassifier를 이용하여 위스콘신 유방암 데이터를 분류할 때, 모델 생성 시 n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42로 설정하였다. 이때 X_train, y_train으로 모델을 학습한 후 X_test에 대한 예측을 수행하고 테스트 정확도를 소수점 넷째 자리까지 출력하기 위해 필요한 코드를 작성하세요.

In [64]:
gb_clf = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
##코드작성##
gb_clf.fit(X_train, y_train)
gb_pred = gb_clf.predict(X_test)
print(f'테스트 정확도: {accuracy_score(y_test, gb_pred):.4f}')


테스트 정확도: 0.8667


## 문제 16. 캐글 신용카드 사기 검출 실습을 통한 불균형 데이터 조정

캐글 신용카드 사기 검출(Credit Card Fraud Detection) 실습에서 레이블이 극도로 불균형한 분포를 가질 때, 소수 클래스(이상 데이터)를 증식시키기 위해 원본 데이터의 피처 값들을 변경하여 새로운 데이터를 생성하는 대표적인 오버 샘플링 기법의 명칭은?

1. 언더 샘플링(Under-Sampling)
2. SMOTE
3. IQR 방식
4. StandScaler


정답 : 2

## 문제 17. IQR 방식을 구현한 이상치 제거

신용카드 사기 데이터에서 사기(Fraud, Class=1)와 정량적 상관관계가 높은 특정 피처(예: V14)의 이상치를 IQR 방식을 통해 제거하는 함수 get_outlier()의 일부이다. 사분위수(Percentile)를 이용해 IQR을 구하고 최솟값과 최댓값을 설정하는 빈칸에 들어갈 올바른 코드는?


In [65]:
import numpy as np
def get_outlier(df=None, column=None, weight=1.5):
    fraud = df[df['Class']==1][column]
    quantile_25 = np.percentile(fraud.values, 25)
    quantile_75 = np.percentile(fraud.values, 75)
    
    iqr = quantile_75 - quantile_25
    iqr_weight = iqr*weight
    lowest_val = quantile_25 -  iqr_weight
    hightest_val = quantile_75 - iqr_weight
    
    outlier_index = fraud[fraud<lowest_val | fraud>hightest_val].index
    return outlier_index

## 문제 18. 스태킹 앙상블 모델 학습 중 과적합 문제 개선

스태킹 앙상블(Stacking Ensemble) 모델을 학습할 때, 기본 스태킹 모델이 가지는 과적합(Overfitting) 문제를 개선하기 위해 최종 메타 모델을 위한 학습용/테스트용 데이터 세트를 생성할 때 활용하는 핵심 기법은?
1. 그리드 탐색 (Grid Search CV)
2. 교차 검증 기반의 스태킹 (CV 세트 기반의 스태킹)
3. 사분위수 편차 방식 (IQR)
4. K최근접 이웃 보간법 (KNN Imputation)


정답 : 2

## 문제 19. 스태킹 앙상블 CV 기반 예측 세트 생성

기본 스태킹 모델이 가지는 과적합(Overfitting) 문제를 해결하기 위해, 교차 검증(Cross Validation) 기반으로 개별 모델의 예측 결과 데이터 세트를 생성하여 최종 메타 모델의 학습용/테스트용 피처 데이터 세트를 만드는 과정에 대한 설명으로 가장 옳은 것은?
1. 개별 모델들이 원본 학습 데이터 전체로 학습한 뒤 테스트 데이터를 예측한 값만 스태킹하여 메타 모델의 학습 피처로 사용한다.
2. 학습 세트를 여러 개의 폴드(Fold)로 분리한 뒤, 검증 폴드로 예측한 결과값들을 쌓아서 최종 메타 모델의 학습 데이터로 사용하고, 테스트 세트의 예측값들은 최종 평균낸 값을 메타 모델의 테스트 데이터로 사용한다.
3. 스태킹 앙상블을 수행할 때는 개별 모델의 하이퍼파라미터 튜닝을 전혀 진행하지 않고 기본값으로만 결합해야만 성능이 보장된다.
4. 메타 모델은 반드시 로지스틱 회귀 모델만 사용해야 하며, 트리 기반 알고리즘은 메타 모델로 사용할 수 없다.


정답 : 2